In [1]:
import requests
import json
import os
import time

### Parliamentary Session Metadata
Loop through all the parliamentary sessions available on LegisInfo (from 35th to 45th).
Download the JSON and export to the /data/canada_parl directory.

In [26]:
parl_sess_list = [
    "35-1",
    "35-2",
    "36-1",
    "36-2",
    "37-1",
    "37-2",
    "37-3",
    "38-1",
    "39-1",
    "39-2",
    "40-1",
    "40-2",
    "40-3",
    "41-1",
    "41-2",
    "42-1",
    "43-1",
    "43-2",
    "44-1",
    "45-1"
]


In [ ]:
output_dir = "../data/canada_parl"
os.makedirs(output_dir, exist_ok=True)

for parl_session in parl_sess_list:
    output_file = os.path.join(output_dir, "can-"+parl_session+".json")
    
    # Skip if file already exists
    if os.path.exists(output_file):
        print(f"Skipping {parl_session} - file already exists")
        continue
    
    response = requests.get("https://www.parl.ca/legisinfo/en/bills/json?parlsession="+parl_session)
    data = response.json()
    with open(output_file, "w", encoding="utf-8") as file:
        json.dump(data, file, indent=4, ensure_ascii=False)

### Canada Bill Text


In [23]:
def fetch_bill_text(parliament, session, bill_type, stage, bill_number):             
    """Fetch bill text from Parliament of Canada DocumentViewer.
    
    Tries to fetch XML first. If empty, falls back to PDF.
    Returns tuple of (text, format) where format is 'xml' or 'pdf'.
    """
    # Try XML first
    xml_url = f"https://www.parl.ca/Content/Bills/{parliament}{session}/{bill_type}/{bill_number}/{bill_number}_{stage}/{bill_number}-e.xml"
    print(f"Fetching XML: {xml_url}")
    try:
        xml_resp = requests.get(xml_url)
        xml_resp.raise_for_status()
        xml_text = xml_resp.text
        
        # Check if XML is not empty
        if xml_text.strip():
            return xml_text, "xml"
    except requests.HTTPError:
        pass

    xml_url = f"https://www.parl.ca/Content/Bills/{parliament}{session}/{bill_type}/{bill_number}/{bill_number}_{stage}/{bill_number}_e.xml"
    print(f"Fetching XML: {xml_url}")
    try:
        xml_resp = requests.get(xml_url)
        xml_resp.raise_for_status()
        xml_text = xml_resp.text
        
        # Check if XML is not empty
        if xml_text.strip():
            return xml_text, "xml"
    except requests.HTTPError:
        pass

    xml_url = f"https://www.parl.ca/Content/Bills/{parliament}{session}/{bill_type}/{bill_number}/{bill_number}_{int(stage)-1}/{bill_number}-e.xml"
    print(f"Fetching XML: {xml_url}")
    try:
        xml_resp = requests.get(xml_url)
        xml_resp.raise_for_status()
        xml_text = xml_resp.text
        
        # Check if XML is not empty
        if xml_text.strip():
            return xml_text, "xml"
    except requests.HTTPError:
        pass
    xml_url = f"https://www.parl.ca/Content/Bills/{parliament}{session}/{bill_type}/{bill_number}/{bill_number}_{int(stage)+1}/{bill_number}-e.xml"
    print(f"Fetching XML: {xml_url}")
    try:
        xml_resp = requests.get(xml_url)
        xml_resp.raise_for_status()
        xml_text = xml_resp.text
        
        # Check if XML is not empty
        if xml_text.strip():
            return xml_text, "xml"
    except requests.HTTPError:
        pass

    xml_url = f"https://www.parl.ca/Content/Bills/{parliament}{session}/{bill_type}/{bill_number}/{bill_number}_{int(stage)-1}/{bill_number}_e.xml"
    print(f"Fetching XML: {xml_url}")
    try:
        xml_resp = requests.get(xml_url)
        xml_resp.raise_for_status()
        xml_text = xml_resp.text
        
        # Check if XML is not empty
        if xml_text.strip():
            return xml_text, "xml"
    except requests.HTTPError:
        pass
    xml_url = f"https://www.parl.ca/Content/Bills/{parliament}{session}/{bill_type}/{bill_number}/{bill_number}_{int(stage)+1}/{bill_number}_e.xml"
    print(f"Fetching XML: {xml_url}")
    try:
        xml_resp = requests.get(xml_url)
        xml_resp.raise_for_status()
        xml_text = xml_resp.text
        
        # Check if XML is not empty
        if xml_text.strip():
            return xml_text, "xml"
    except requests.HTTPError:
        pass
    
    pdf_url = f"https://www.parl.ca/Content/Bills/{parliament}{session}/{bill_type}/{bill_number}/{bill_number}_{stage}/{bill_number}_{stage}.pdf"
    print(f"XML empty or failed, fetching PDF: {pdf_url}")
    pdf_resp = requests.get(pdf_url)
    pdf_resp.raise_for_status()
    return pdf_resp.content, "pdf"

In [28]:
# Example: fetch text for all bills in a session                              
                 
stage_map = {
    "First reading":  "1",
    "Second reading": "2",
    "Third reading":  "3",
    "Royal assent":   "4",
}

for session in parl_sess_list:
    session_data = json.load(open(f"../data/canada_parl/can-{session}.json"))
    os.makedirs(f"../data/canada_bill_text/{session}", exist_ok=True)  
    for bill in session_data:
        if bill["LatestBillTextTypeId"] == 0:                                     
            continue  # no text available                                         
        bill_num = bill["BillNumberFormatted"]
        parl = bill["ParliamentNumber"]                                           
        sess = bill["SessionNumber"]
        stage = stage_map.get(bill["LatestCompletedMajorStageEn"], "1")
        bill_type = "Government" if (bill["BillTypeEn"] == "House Government Bill" or bill["BillTypeEn"] == "Senate Government Bill") else "Private"                             
        out_path_base = f"../data/canada_bill_text/{parl}-{sess}/{bill_num}"
        xml_path = out_path_base + ".xml"
        pdf_path = out_path_base + ".pdf"
        
        # Skip if either format already exists
        if os.path.exists(xml_path) or os.path.exists(pdf_path):                                              
            continue
        try:                                                                      
            content, fmt = fetch_bill_text(parl, sess, bill_type, stage, bill_num)
            
            # Save with appropriate extension
            if fmt == "xml":
                with open(xml_path, "w", encoding="utf-8") as f:
                    f.write(content)
            else:  # PDF
                with open(pdf_path, "wb") as f:
                    f.write(content)
                    
            time.sleep(2)  # be polite to the API                             
        except requests.HTTPError as e:                                           
            print(f"Error fetching bill {bill_num}: {e}")
            continue

Fetching XML: https://www.parl.ca/Content/Bills/351/Government/C-7/C-7_2/C-7-e.xml
Fetching XML: https://www.parl.ca/Content/Bills/351/Government/C-7/C-7_2/C-7_e.xml
Fetching XML: https://www.parl.ca/Content/Bills/351/Government/C-7/C-7_1/C-7-e.xml
Fetching XML: https://www.parl.ca/Content/Bills/351/Government/C-7/C-7_3/C-7-e.xml
Fetching XML: https://www.parl.ca/Content/Bills/351/Government/C-7/C-7_1/C-7_e.xml
Fetching XML: https://www.parl.ca/Content/Bills/351/Government/C-7/C-7_3/C-7_e.xml
XML empty or failed, fetching PDF: https://www.parl.ca/Content/Bills/351/Government/C-7/C-7_2/C-7_2.pdf
Error fetching bill C-7: 404 Client Error: Not Found for url: https://www.parl.ca/ErrorPage/Default.aspx?Url=https%3a%2f%2fwww.parl.ca%2fContent%2fBills%2f351%2fGovernment%2fC-7%2fC-7_2%2fC-7_2.pdf&StatusCode=404
Fetching XML: https://www.parl.ca/Content/Bills/351/Government/C-52/C-52_2/C-52-e.xml
Fetching XML: https://www.parl.ca/Content/Bills/351/Government/C-52/C-52_2/C-52_e.xml
Fetching XML:

ConnectionError: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))